In [58]:
import torch
from torch import nn, optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import random_split
from datetime import datetime
import numpy as np
import copy



# Getting the same results with train and train_manual_update
- Write torch.manual_seed(42) at the beginning of your notebook.
- Write torch.set_default_dtype(torch.double) at the beginning of your notebook to alleviate precision errors

In [59]:
torch.manual_seed(42)
torch.set_default_dtype(torch.double)


# Tasks
Load, analyse and preprocess the CIFAR-10 dataset. Split it into 3
datasets: training, validation and test. Take a subset of these datasets
by keeping only 2 labels: bird and plane

In [ ]:
def load_cifar(train_val_split=0.9, data_path='../data/', preprocessor=None):
    if preprocessor  is None:
        preprocessor = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4915, 0.4823, 0.4468),
                                 (0.2470, 0.2435, 0.2616))
        ])
    
    # load datasets
    data_train_val = datasets.CIFAR10(
        data_path,       
        train=True,      
        download=True,  
        transform=preprocessor)

    data_test = datasets.CIFAR10(
        data_path, 
        train=False,
        download=True,
        transform=preprocessor)

    # Split the data into training and validation sets based on the train_val_split ratio
    n_train = int(len(data_train_val) * train_val_split)
    n_val =  len(data_train_val) - n_train

    data_train, data_val = random_split(
        data_train_val, 
        [n_train, n_val],
        generator=torch.Generator().manual_seed(42)
    )

    print(f"Train size: {len(data_train)}")
    print(f"Validation size: {len(data_val)}")
    print(f"Test size: {len(data_test)}")

    label_map = {0: 0, 2: 1}
    class_names = ['airplane', 'bird']

    # For each part of dataset, keep only airplanes and birds. it means that we create new datasets 
    #(cifar2_train, cifar2_val, and cifar2_test) by filtering the original datasets (cifar10_train, cifar10_val, 
    # and cifar10_test) to only include the selected classes (0 and 2) and mapping their labels using label_map.
    #training data set:
    cifar2_train = [(img, label_map[label]) for img, label in data_train if label in [0, 2]]
    cifar2_val = [(img, label_map[label]) for img, label in data_val if label in [0, 2]]
    cifar2_test = [(img, label_map[label]) for img, label in data_test if label in [0, 2]]

    print(f"Train size: {len(cifar2_train)}")
    print(f"Validation size: {len(cifar2_val)}")
    print(f"Test size: {len(cifar2_test)}")

    
    
    return (cifar2_train, cifar2_val, cifar2_test)

cifar2_train, cifar2_val, cifar2_test = load_cifar()


def compute_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in loader:
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
            total += data.size(0)
    return correct / total

Files already downloaded and verified


/opt/miniconda3/envs/INF265/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Files already downloaded and verified
Train size: 45000
Validation size: 5000
Test size: 10000
Train size: 8980
Validation size: 1020
Test size: 2000


Write a MyMLP class that implements a MLP in PyTorch (so only fully
connected layers) such that:
    
    - The input dimension is 768(= 16 ∗ 16 ∗ 3) and the output dimension is 2 (for the 2 classes).
    - The hidden layers have respectively 128 and 32 hidden units.
    - All activation functions are ReLU. The last layer has no activation function since the cross-entropy loss already includes a softmax activation
function.

In [61]:
class MyNet(nn.Module):
    def __init__(self):
        super(MyNet, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(3072, 512),
            nn.ReLU(),  
            nn.Linear(512, 128),
            nn.ReLU(),  
            nn.Linear(128, 32),
            nn.ReLU(), 
            nn.Linear(32, 2)
        )

    def forward(self, x):
        x = torch.flatten(x, 1)  # (B, 3, 32, 32) -> (B, 3072)
        return self.network(x)

Write a train(n_epochs, optimizer, model, loss_fn, train_loader) function that trains model for n_epochs epochs given an optimizer optimizer, a loss function loss_fn and a dataloader train_loader.

In [62]:
def train(n_epochs, optimizer, model, loss_fn, train_loader):
    model.train()
    losses = []
    for epoch in range(n_epochs):
        for data, target in train_loader:
            data = data.view(data.size(0), -1)  # flatten
            output = model(data)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(output, target)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")
    return model, losses

Write a similar function train manual_update that has no optimizer parameter, but a learning rate lr parameter instead and that manually updates each trainable parameter of model using equation (2). Do not forget to zero out all gradients after each iteration. 

Train 2 instances of MyMLP, one using train and the other using train_manual_update (use the same parameter values for both models). Compare their respective training losses. To get exactly the same results with both functions, see section 3.3

In [63]:
def train_manual_update(n_epochs, model, loss_fn, train_loader, lr=1e-2, momentum_coeff=0.0, weight_decay=0.0):
    model.train()
    velocity = {id(p): torch.zeros_like(p) for p in model.parameters() if p.requires_grad}
    losses = []

    for epoch in range(n_epochs):
        for data, target in train_loader:
            # forward
            data = data.view(data.size(0), -1)  # flatten if your model expects vectors
            output = model(data)
            loss = loss_fn(output, target)

            # backward
            loss.backward()

            # manual parameter update (equation 2: SGD step; extended with momentum + wd)
            with torch.no_grad():
                for p in model.parameters():
                    if not p.requires_grad:
                        continue
                    if p.grad is None:
                        continue

                    grad = p.grad

                    # weight decay (L2 regularization) in SGD: grad <- grad + wd * p
                    if weight_decay != 0.0:
                        grad = grad.add(p, alpha=weight_decay)

                    if momentum_coeff != 0.0:
                        v = velocity[id(p)]
                        v.mul_(momentum_coeff).add_(grad)   # v = mu*v + grad
                        p.add_(v, alpha=-lr)                # p = p - lr*v
                    else:
                        p.add_(grad, alpha=-lr)             # p = p - lr*grad

                    # zero grad for next iteration
                    p.grad = None

            losses.append(loss.item())

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}, Loss: {losses[-1]:.4f}")
        

    return model, losses

# Comparison of train vs train_manual_update and their respective losses

In [64]:
def comparison(weight_decay=0.0, momentum_coeff=0.0):
    model = MyNet()
    model2 = copy.deepcopy(model)

    optimizer = optim.SGD(model.parameters(), lr=1e-2, weight_decay=weight_decay, momentum=momentum_coeff)
    loss_fn = nn.CrossEntropyLoss()
    train_loader = torch.utils.data.DataLoader(cifar2_train, batch_size=64, shuffle=False)
    epochs = 5

    model, losses = train(epochs, optimizer, model, loss_fn, train_loader)
    model2, losses2 = train_manual_update(epochs, model2, loss_fn, train_loader, weight_decay=weight_decay, momentum_coeff=momentum_coeff)

    comparison = losses == losses2
    return comparison

In [65]:
comparison1 = comparison()
comparison2 = comparison(weight_decay=1e-4)
comparison3 = comparison(weight_decay=1e-4, momentum_coeff=0.9)
print("Comparison without weight decay and momentum:", comparison1)
print("Comparison with weight decay:", comparison2)
print("Comparison with weight decay and momentum:", comparison3)

Comparison without weight decay and momentum: True
Comparison with weight decay: True
Comparison with weight decay and momentum: True


# Pipeline